In [1]:
import numpy as np # used for array operations
import brian2 as b2 # used for neural simulation
from brian2 import NeuronGroup, Synapses, PoissonGroup, SpikeMonitor, run, ms, Hz, PoissonInput, clear_cache, device, get_unit, second
from brian2 import *
from scipy.stats import poisson, binom # used for stats (mean, s.d.)
import matplotlib.pyplot as plt # used for plotting
import json # to serialize
b2.prefs.codegen.target = "numpy" # set Brian2 to use numpy backend
import random
from scipy import stats

In [2]:
def display_spike_summary(spike_event, spike_time, num_neurons, num_samples):
    """
    Creates a formatted summary table of spike events and times
    Limited to maximum of 10 neurons and 10 samples
    """
    # Limit to maximum of 10 neurons and 10 samples
    display_neurons = min(num_neurons, 3)
    display_samples = min(num_samples, 2)
    total_rows = display_neurons * display_samples
    
    print("\nSummary of Poisson Process Spike Trains")
    print("=" * 100)
    print(f"{'Neuron':<10} {'Sample':<10} {'First 3 Spike Times (ms)':<40}")
    print("-" * 100)
    
    for idx in range(total_rows):
        neuron_num = idx % display_neurons
        sample_num = idx // display_neurons
        
        
        # Get first 5 spike times
        first_spikes = str(spike_time[idx][:3])
        
        print(f"{neuron_num:<10} {sample_num:<10} {first_spikes:<40}")
    
    print("=" * 100)

In [3]:
def independent_poisson_processes(num_neurons, rate, time, num_samples):
    """
    Description:
    -----------
    Generates independent Poisson process spike trains for multiple neurons.
    
    Parameters:
    -----------
    num_neurons : Number of neurons to simulate
    rate : Firing rate of neurons in Hz
    time : Duration of simulation in milliseconds
    num_samples : Number of samples to generate for each neuron
    
    Returns:
    -------
    spike_train : returns the spike event and timing for each neuron
    
    Note:
    -----
    - Uses two for loops so I can iterate each neuron and their samples.
    - I want to implement read() also but just putting read() is causing an error
    - Probably because I need it to read to something instead of just read()
    """
    spike_trains = []
    P = PoissonGroup(num_neurons, rate * b2.Hz)
    spike_monitor = SpikeMonitor(P)
    defaultclock.dt = 0.1 * ms
    
    store()

    for sample in range(num_samples):
        restore()
        run(time * ms)
            
        spike_trains_dict = spike_monitor.spike_trains() # this is the guy that makes ms and seconds in the output
        spike_trains.append(spike_trains_dict)
    
    return spike_trains

In [4]:
def count_spikes(binary_data):
    """
    Counts cumulative spikes over time for multiple neurons across multiple samples.
    
    Parameters:
    -----------
    binary_data : List of lists containing numpy arrays of binary spike data
                 where 1 indicates a spike and 0 indicates no spike
    
    Returns:
    --------
    counts : List of lists containing numpy arrays of cumulative spike counts
    """
    counts = []
    
    # For each sample
    for sample in binary_data:
        sample_counts = []
        
        # For each neuron in the sample
        for neuron_data in sample:
            # Use cumsum to count spikes cumulatively
            spike_count = np.cumsum(neuron_data)
            sample_counts.append(spike_count)
            
        counts.append(sample_counts)
    
    return counts

In [5]:
def calculate_theoretical_mean(rate, time):
    theoretical_mean = np.zeros((2, time))
    for i in range(time):
        theoretical_mean[0, i] = (rate * (i)) / 1000 
        theoretical_mean[1, i] = (rate * (i)) / 1000 
    return theoretical_mean

In [6]:
def calculate_theoretical_std_dev(theoretical_means):
    return np.sqrt(theoretical_means)

In [7]:
def empirical_means(count):
    total = 0

    for sample in count:
        total += sample

    # np.mean(total) test this with the list of samples
    mean = total / num_samples
    return mean

In [8]:
def std_dev_empirical_mean(empirical_mean, num_samples, count):
    # Initialize the sum of squared differences
    sum_squared_diff = 0

    # Calculate the sum of squared differences
    for sample in count:
        sum_squared_diff += (sample - empirical_mean) ** 2

    # Calculate the variance
    variance = sum_squared_diff / (num_samples - 1)

    # Calculate the standard deviation
    std_dev = np.sqrt(variance)

    return std_dev

In [ ]:
def variance_of_residuals(observed, expected):
    residuals = observed - expected
    return np.var(residuals, ddof=1)  # ddof=1 for sample variance

In [9]:
def plot_count_neuron1_vs_time(count, num_samples):
    plt.figure(figsize=(10,6))
    for i in range(num_samples):
        plt.plot(count[i][0], label=f'Sample {i}')
    #mean_neuron1 = np.mean([counting_process_nd[i][0] for i in range(num_samples)], axis=0)
    #plt.plot(mean_neuron1, label='Mean', color='black', linewidth=2)
    plt.xlabel('Time')
    plt.ylabel('Count of Neuron 1')
    plt.title('Count of Neuron 1 Over Time')
    plt.ylim(0, None)  # Set y-axis lower limit to 0
    plt.show()

In [10]:
def plot_centered_staircase_3d(count, num_samples, theoretical_means, theoretical_std_dev):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Iterate through the number of samples
    for i in range(num_samples):
        # Ensure that counting_process_nd_result[i] is structured correctly
        neuron1_data = count[i][0] - theoretical_means[0]
        neuron2_data = count[i][1] - theoretical_means[1]
        
        # Plot the data, by subtracting by the theoretical_means, i should be centering the data but it is not doing that ....
        ax.plot(range(len(neuron1_data)), neuron2_data, neuron1_data)

    # Calculate mean centered data for both neurons 
#    mean_centered_neuron1 = np.mean([(counting_process_nd_result[i][0] - theoretical_means[0]) / theoretical_std_dev[0] for i in range(num_samples)], axis=0)
#    mean_centered_neuron2 = np.mean([(counting_process_nd_result[i][1] - theoretical_means[1]) / theoretical_std_dev[1] for i in range(num_samples)], axis=0)

    # Plot mean centered data
#    ax.plot(range(len(mean_centered_neuron1)), mean_centered_neuron2, mean_centered_neuron1, color='black', linewidth=2)

    ax.set_xlabel('Time')
    ax.set_ylabel('Centered Count of Neuron 2')
    ax.set_zlabel('Centered Count of Neuron 1', rotation=90)
    ax.set_title('Centered Count of Neuron 1 vs Centered Count of Neuron 2 vs Time')

    ax.set_xlim(0, max(len(count[i][0]) for i in range(num_samples)))
#    max_val = max(max((counting_process_nd_result[i][1] - theoretical_means[1]) / theoretical_std_dev[1]) for i in range(num_samples))
#    ax.set_ylim(-max_val, max_val)
#    max_val = max(max((counting_process_nd_result[i][0] - theoretical_means[0]) / theoretical_std_dev[0]) for i in range(num_samples))
#    ax.set_zlim(-max_val, max_val)

    plt.show()


In [11]:
def plot_standardized_staircase_3d(count, num_samples, theoretical_means, theoretical_std_dev, t0):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Iterate through the number of samples
    for i in range(num_samples):
        # Ensure that counting_process_nd_result[i] is structured correctly
        neuron1_data = (count[i][0] - theoretical_means[0]) / (theoretical_std_dev[0])
        neuron2_data = (count[i][1] - theoretical_means[1]) / (theoretical_std_dev[1])
        
        # Plot the data, len(neuron1_data) should be = to len(neuron2_data) so either can be used
        ax.plot(range(t0,len(neuron1_data)), neuron2_data[t0:], neuron1_data[t0:])

    # Calculate mean standardized data for both neurons
#    mean_standardized_neuron1 = np.mean([counting_process_nd_result[i][0] / theoretical_std_dev[0] for i in range(num_samples)], axis=0)
#    mean_standardized_neuron2 = np.mean([counting_process_nd_result[i][1] / theoretical_std_dev[1] for i in range(num_samples)], axis=0)

    # Plot mean standardized data
#     ax.plot(range(len(mean_standardized_neuron1)), mean_standardized_neuron2, mean_standardized_neuron1, color='black', linewidth=2)

    ax.set_xlabel('Time')
    ax.set_ylabel('Standardized Count of Neuron 2')
    ax.set_zlabel('Standardized Count of Neuron 1', rotation=90)
    ax.set_title('Standardized Count of Neuron 1 vs Standardized Count of Neuron 2 vs Time')

#    ax.set_xlim(100, max(len(counting_process_nd_result[i][0]) for i in range(num_samples)))
#    ax.set_ylim(min(min(counting_process_nd_result[i][1] / theoretical_std_dev[1]) for i in range(num_samples)),
#                max(max(counting_process_nd_result[i][1] / theoretical_std_dev[1]) for i in range(num_samples)))
#    ax.set_zlim(min(min(counting_process_nd_result[i][0] / theoretical_std_dev[0]) for i in range(num_samples)),
#                max(max(counting_process_nd_result[i][0] / theoretical_std_dev[0]) for i in range(num_samples)))

    plt.show()


In [12]:
def mean_trajectory(count):
    """Calculate mean trajectory for centered and standardized data"""
    mean_trajectory = np.mean(count, axis=0)
    std_trajectory = np.std(count, axis=0)
    
    # Center and standardize
    centered = mean_trajectory - np.mean(mean_trajectory)
    standardized = (mean_trajectory - np.mean(mean_trajectory)) / std_trajectory
    
    return centered, standardized

In [13]:
def confidence_intervals(mean, std_dev, num_samples, confidence_level=0.95):
    """
    Calculate confidence intervals for neural data.
    
    Parameters:
    -----------
    mean : Mean values of your data. Could be mean spike counts/rates
    std_dev : Standard deviation of your data
    num_samples : Number of trials/samples
    confidence_level : Desired confidence level (default 0.95 for 95% confidence)
    
    Returns:
    --------
    lower_bound : Lower confidence bound
    upper_bound : Upper confidence bound
    """
    
    t_value = stats.t.ppf((1 + confidence_level) / 2, df=num_samples-1)
    
    # Calculate standard error
    standard_error = std_dev / np.sqrt(num_samples)
    
    # Calculate margin of error
    margin_of_error = t_value * standard_error
    
    # Calculate bounds
    lower_bound = mean - margin_of_error
    upper_bound = mean + margin_of_error
    
    return lower_bound.tolist(), upper_bound.tolist()

In [14]:
def sim(g, nu_ext_over_nu_thr, time, num_neurons, num_samples):
    """
    Parameters:
    -----------
    g : float
        Relative inhibitory to excitatory synaptic strength
    nu_ext_over_nu_thr : float
        Ratio of external stimulus rate to threshold rate
    sim_time : brian2.units.fundamentalunits.Quantity
        Simulation time in milliseconds
    ax_spikes : matplotlib.axes.Axes
        Axes to plot spikes on
    ax_rates : matplotlib.axes.Axes
        Axes to plot rates on
    rate_tick_step : float
        Step size for rate axis ticks
    """
    # # Network parameters
    # N_E = 1000
    # gamma = 0.25
    # N_I = round(gamma * N_E)
    # N = N_E + N_I
    # epsilon = 0.1
    # C_E = epsilon * N_E
    # C_ext = C_E

    # # Neuron parameters (all in Brian2 units)
    # tau = 20 * ms
    # theta = 20 * mV
    # V_r = 10 * mV
    # tau_rp = 2 * ms

    # # Synapse parameters
    # J = 0.1 * mV
    # D = 1.5 * ms

    # External stimulus
    nu_thr = theta / (J * C_E * tau)
    nu_ext = nu_ext_over_nu_thr * nu_thr

    # Set simulation timestep
    defaultclock.dt = 0.1 * ms

    # Create neuron groups
    neurons = NeuronGroup(N,
                         '''
                         dv/dt = -v/tau : volt (unless refractory)
                         ''',
                         threshold='v > theta',
                         reset='v = V_r',
                         refractory=tau_rp,
                         method='exact')

    # Split into excitatory and inhibitory populations
    excitatory_neurons = neurons[:N_E]
    inhibitory_neurons = neurons[N_E:]

    # Create synapses
    exc_synapses = Synapses(excitatory_neurons, neurons, 
                           on_pre='v_post += J',
                           delay=D)
    exc_synapses.connect(p=epsilon)

    inhib_synapses = Synapses(inhibitory_neurons, neurons, 
                             on_pre='v_post += -g*J',
                             delay=D)
    inhib_synapses.connect(p=epsilon)

    # Add external input
    external_poisson_input = PoissonInput(neurons, 'v', 
                                        N=C_ext,
                                        rate=nu_ext,
                                        weight=J)

    # Set up monitors
    rate_monitor_exc = PopulationRateMonitor(excitatory_neurons)
    rate_monitor_inh = PopulationRateMonitor(inhibitory_neurons)
    spike_monitor_exc = SpikeMonitor(excitatory_neurons[:num_neurons])
    spike_monitor_inh = SpikeMonitor(inhibitory_neurons[:num_neurons])

    # Create and run network
    net = Network(neurons, exc_synapses, inhib_synapses, 
                 external_poisson_input,
                 rate_monitor_exc, rate_monitor_inh,
                 spike_monitor_exc, spike_monitor_inh)

    #####################################
    excitatory_spikes = []
    inhibitory_spikes = []
    net.store()
    for sample in range(num_samples):
        net.restore()
        
        # Run the simulation
        net.run(time * ms, report=None)
        
        # Collect spike trains for this sample
        excitatory_spikes.append(spike_monitor_exc.spike_trains())
        inhibitory_spikes.append(spike_monitor_inh.spike_trains())
    
    #####################################

    # Get time ranges for plotting (in ms)
    # t_start = params["t_range"][0] * ms
    # t_end = params["t_range"][1] * ms

    # # Plot spikes
    # ax_spikes.plot(spike_monitor_exc.t/ms, 
    #                spike_monitor_exc.i,
    #                '|', color='blue', label='Excitatory')
    # ax_spikes.plot(spike_monitor_inh.t/ms,
    #                spike_monitor_inh.i + 25,
    #                '|', color='red', label='Inhibitory')

    # # Plot rates
    # ax_rates.plot(rate_monitor_exc.t/ms,
    #               rate_monitor_exc.rate/Hz,
    #               color='blue', label='Excitatory')
    # ax_rates.plot(rate_monitor_inh.t/ms,
    #               rate_monitor_inh.rate/Hz,
    #               color='red', label='Inhibitory')

    # # Configure plots
    # ax_spikes.set_yticks([])
    # ax_spikes.legend(loc='upper right')
    # ax_rates.legend(loc='upper right')
    
    # ax_spikes.set_xlim(t_start/ms, t_end/ms)
    # ax_rates.set_xlim(t_start/ms, t_end/ms)
    # ax_rates.set_ylim(*params["rate_range"])
    # ax_rates.set_xlabel("t [ms]")
    
    # ax_rates.set_yticks(np.arange(
    #     params["rate_range"][0],
    #     params["rate_range"][1] + rate_tick_step,
    #     rate_tick_step
    # ))

    # plt.subplots_adjust(hspace=0)

    return {
        'excitatory': {
            'spike_times': spike_monitor_exc.t,  # Keep Brian2 units
            'spike_indices': spike_monitor_exc.i,
            'rate_times': rate_monitor_exc.t,
            'rate_values': rate_monitor_exc.rate,
            'spike_trains': excitatory_spikes
        },
        'inhibitory': {
            'spike_times': spike_monitor_inh.t,  # Keep Brian2 units
            'spike_indices': spike_monitor_inh.i,
            'rate_times': rate_monitor_inh.t,
            'rate_values': rate_monitor_inh.rate,
            'spike_trains': inhibitory_spikes
        }
    }
# parameters = {
#     "C": {
#         "g": 7, # g is a good control for manipulating the firing rate
#         "nu_ext_over_nu_thr": 2,
#         "t_range": [1000, 1200],
#         "rate_range": [0, 200],
#         "rate_tick_step": 50,
#     },
# }

# for panel, params in parameters.items():
#     fig = plt.figure(figsize=(4, 5))
#     fig.suptitle(panel)

#     gs = fig.add_gridspec(ncols=1, nrows=2, height_ratios=[4, 1])
#     ax_spikes, ax_rates = gs.subplots(sharex="col")

#     results = sim(
#         params["g"],
#         params["nu_ext_over_nu_thr"],
#         params["t_range"][1] * ms,
#         ax_spikes,
#         ax_rates,
#         params["rate_tick_step"],
#     )
    
#     # Print statistics for both populations
#     for pop_type in ['excitatory', 'inhibitory']:
#         print(f"\n{pop_type.capitalize()} population:")
#         print(f"Spike Times (first 10): {results[pop_type]['spike_times'][:10]}")
#         print(f"Spike Indices (first 10): {results[pop_type]['spike_indices'][:10]}")
#         print(f"Mean firing rate: {np.mean(results[pop_type]['rate_values'])} Hz")
#         print(f"Number of spikes: {len(results[pop_type]['spike_times'])}")

# plt.show()

In [15]:
def create_connections(excitatory_data, inhibitory_data, p_connection=0.1, 
                      weight_exc=0.1*mV, weight_inh=-0.5*mV):
    """
    Create synaptic connections between neurons based on their spike train data.
    
    Parameters:
    -----------
    excitatory_data : dict
        Dictionary containing excitatory neuron data
    inhibitory_data : dict
        Dictionary containing inhibitory neuron data
    p_connection : float
        Connection probability (default 0.1)
    weight_exc : brian2.units.fundamentalunits.Quantity
        Weight for excitatory synapses
    weight_inh : brian2.units.fundamentalunits.Quantity
        Weight for inhibitory synapses
    
    Returns:
    --------
    tuple
        (exc_to_exc, exc_to_inh, inh_to_exc, inh_to_inh) Synapses objects
    """
    
    # Get number of neurons from spike trains
    N_exc = len(excitatory_data['spike_trains'])
    N_inh = len(inhibitory_data['spike_trains'])
    
    # Create neuron groups
    excitatory_neurons = NeuronGroup(N_exc,
                                   '''dv/dt = -v/tau : volt (unless refractory)
                                      tau : second''',
                                   threshold='v > 20*mV',
                                   reset='v = 0*mV',
                                   refractory=2*ms,
                                   method='exact')
    
    inhibitory_neurons = NeuronGroup(N_inh,
                                   '''dv/dt = -v/tau : volt (unless refractory)
                                      tau : second''',
                                   threshold='v > 20*mV',
                                   reset='v = 0*mV',
                                   refractory=2*ms,
                                   method='exact')
    
    # Create synapses with random connectivity and no self-connections
    
    # Excitatory to Excitatory
    exc_to_exc = Synapses(excitatory_neurons, excitatory_neurons,
                         model='w : volt',
                         on_pre='v_post += w')
    exc_to_exc.connect(condition='i != j', p=p_connection)  # No self-connections
    exc_to_exc.w = weight_exc
    
    # Excitatory to Inhibitory
    exc_to_inh = Synapses(excitatory_neurons, inhibitory_neurons,
                         model='w : volt',
                         on_pre='v_post += w')
    exc_to_inh.connect(p=p_connection)
    exc_to_inh.w = weight_exc
    
    # Inhibitory to Excitatory
    inh_to_exc = Synapses(inhibitory_neurons, excitatory_neurons,
                         model='w : volt',
                         on_pre='v_post += w')
    inh_to_exc.connect(p=p_connection)
    inh_to_exc.w = weight_inh
    
    # Inhibitory to Inhibitory
    inh_to_inh = Synapses(inhibitory_neurons, inhibitory_neurons,
                         model='w : volt',
                         on_pre='v_post += w')
    inh_to_inh.connect(condition='i != j', p=p_connection)  # No self-connections
    inh_to_inh.w = weight_inh
    
    # Print connection statistics
    print("\nConnection statistics:")
    print(f"E→E connections: {len(exc_to_exc.w)} "
          f"({len(exc_to_exc.w)/(N_exc**2)*100:.1f}% connected)")
    print(f"E→I connections: {len(exc_to_inh.w)} "
          f"({len(exc_to_inh.w)/(N_exc*N_inh)*100:.1f}% connected)")
    print(f"I→E connections: {len(inh_to_exc.w)} "
          f"({len(inh_to_exc.w)/(N_inh*N_exc)*100:.1f}% connected)")
    print(f"I→I connections: {len(inh_to_inh.w)} "
          f"({len(inh_to_inh.w)/(N_inh**2)*100:.1f}% connected)")
    
    return exc_to_exc, exc_to_inh, inh_to_exc, inh_to_inh

In [16]:
def convert_spike_trains_to_binary(spike_trains_dict, time, num_neurons):
    """
    Convert Brian2 spike trains to binary arrays.
    
    Parameters:
    -----------
    spike_trains_dict : dict
        Dictionary of spike trains from Brian2 SpikeMonitor
    time : int
        Duration of simulation in ms
    num_neurons : int
        Number of neurons to convert
    
    Returns:
    --------
    spike_events : list
        List of binary arrays (1=spike, 0=no spike) for each neuron
    """
    spike_events = []
    
    for n in range(num_neurons):
        events = np.zeros(int(time))
        if n in spike_trains_dict:
            # Convert spike times to milliseconds and to indices
            spike_times = spike_trains_dict[n]
            spike_indices = (spike_times/ms).astype(int)
            # Only include spikes within the time window
            valid_indices = spike_indices[spike_indices < time]
            events[valid_indices] = 1
        spike_events.append(events)
    
    return spike_events

In [17]:
def save_spike_trains(spike_trains, filename):
    """
    Save spike trains to a JSON file.
    
    Parameters:
    -----------
    spike_trains : list of dict
        List of dictionaries containing spike trains
    filename : str
        Name of the file to save to (should end in .json)
    """
    serializable_spike_trains = []
    for sample in spike_trains:
        sample_dict = {}
        for neuron_id, times in sample.items():
            sample_dict[str(neuron_id)] = (times/second).tolist()
        serializable_spike_trains.append(sample_dict)
    
    with open(filename, 'w') as f:
        json.dump(serializable_spike_trains, f)
    
    print(f"Spike trains saved to {filename}")


In [18]:
def load_spike_trains(filename):
    """
    Load spike trains from a JSON file.
    
    Parameters:
    -----------
    filename : str
        Name of the file to load from
        
    Returns:
    --------
    spike_trains : list of dict
        List of dictionaries where each dictionary contains spike times for different neurons with Brian2 units
    """
    with open(filename, 'r') as f:
        loaded_spike_trains = json.load(f)
    
    spike_trains = []
    for sample in loaded_spike_trains:
        sample_dict = {}
        for neuron_id, times in sample.items():
            sample_dict[int(neuron_id)] = np.array(times) * second
        spike_trains.append(sample_dict)
    
    return spike_trains


In [19]:
def standardize_units(spike_trains):
    """
    Converts all spike times in a spike trains dictionary to unitless values in seconds.

    Parameters:
    -----------
    spike_trains : list of dict
        List where each dictionary contains spike times for different neurons
        
    Returns:
    --------
    standardized_trains : list of dict
        List of dictionaries with all times converted to unitless values in seconds
    """
    standardized_trains = []
    
    for sample in spike_trains:
        converted_sample = {}
        for neuron_id, times in sample.items():
            converted_sample[neuron_id] = times/second
        standardized_trains.append(converted_sample)
    
    return standardized_trains


In [20]:
def spikes_to_binary(spike_trains, num_neurons, time):
    """
    Convert spike times to binary sequences
    
    Parameters:
    -----------
    spike_trains : list of dictionaries
        Each dictionary contains spike times for neurons
    num_neurons : int
        Number of neurons in the data
    time : int
        Duration of recording in milliseconds
    time_resolution : float
        Time bin size in seconds (default=0.001 for millisecond resolution)
        
    Returns:
    --------
    binary_data : list of lists
        Each inner list contains numpy arrays (one per neuron) of 0s and 1s
    """
    binary_data = []
    
    for sample in spike_trains:
        sample_data = []
        
        # Process each neuron
        for neuron in range(num_neurons):
            # Create zero array with length equal to time parameter
            spike_array = np.zeros(time)
            
            # Get spike times for this neuron
            spike_times = sample[neuron]
            
            spike_indices = (spike_times * 1000).astype(int)
            
            # Set spikes to 1
            for idx in spike_indices:
                if 0 <= idx < time:  # Ensure index is within bounds
                    spike_array[idx] = 1
                    
            sample_data.append(spike_array)
            
        binary_data.append(sample_data)
    
    return binary_data
